In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = (SparkSession.builder
    .appName("UrgenceNet-Partie4-Streaming")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

event_schema = StructType([
    StructField("visit_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("facility_id", StringType()),
    StructField("city", StringType()),
    StructField("status", StringType()),
    StructField("priority", IntegerType()),
    StructField("wait_minutes", IntegerType()),
    StructField("department", StringType()),
    StructField("desk_id", StringType()),
    StructField("transfer_to", StringType()),
    StructField("event_time", TimestampType()),
])

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 23:02:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "urgences")
    .option("startingOffsets", "latest")
    .load())

events = (raw.select(F.from_json(F.col("value").cast("string"), event_schema).alias("e"))
             .select("e.*"))

raw.printSchema()
events.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: integer (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- desk_id: string (nullable = true)
 |-- transfer_to: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [3]:
q_console = (events.writeStream
    .queryName("console_raw")
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 5)
    .trigger(processingTime="10 seconds")
    .start())

26/11/05 23:03:32 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-dd2d2815-9d96-4b07-8142-25e094cfcc60. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/05 23:03:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [4]:
bronze = raw.select(
    F.col("key").cast("string").alias("key"),
    F.col("value").cast("string").alias("value"),      # JSON brut, tel que lu dans Kafka
    "topic", "partition", "offset",
    F.col("timestamp").alias("kafka_timestamp"))

q_bronze = (bronze.writeStream
    .queryName("bronze_raw")
    .format("json")
    .outputMode("append")
    .option("path", "s3a://lake/bronze/urgences_raw")
    .option("checkpointLocation", "s3a://lake/checkpoints/bronze_urgences_raw")
    .trigger(processingTime="30 seconds")
    .start())

26/11/05 23:03:37 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/05 23:03:43 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/05 23:03:53 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [5]:
counts = (events.filter(F.col("status") == "TRIAGED")
                .groupBy("city", "priority").count()
                .orderBy("city", "priority"))

q_counts = (counts.writeStream
    .queryName("counts_city_priority")
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 25)
    .trigger(processingTime="15 seconds")
    .start())

26/11/05 23:04:06 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/05 23:04:08 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3cfe89f4-5a39-4a2d-a68f-9cd109966599. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/05 23:04:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [6]:
wm = (events.filter(F.col("status") == "TRIAGED")
            .withWatermark("event_time", "10 minutes")
            .groupBy(F.window("event_time", "10 minutes"), "city", "priority")
            .count())

q_wm = (wm.writeStream
    .queryName("watermark_demo")
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 8)
    .trigger(processingTime="15 seconds")
    .start())

26/11/05 23:04:11 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/11/05 23:04:17 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-46df5e6d-9244-46f1-a081-ab8a0b467e09. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/05 23:04:17 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [7]:
for q in spark.streams.active:
    print(q.name, "|", q.status["message"])

p = q_wm.lastProgress
if p:
    print("eventTime :", p["eventTime"])
    print("Lignes ignorées (en retard) :",
          [s.get("numRowsDroppedByWatermark") for s in p["stateOperators"]])

counts_city_priority | Processing new data
bronze_raw | Processing new data
watermark_demo | Getting offsets from KafkaV2[Subscribe[urgences]]
console_raw | Processing new data


26/11/05 23:04:20 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
[Stage 1:>    (0 + 3) / 3][Stage 2:>    (0 + 1) / 3][Stage 4:>    (0 + 0) / 3]  

-------------------------------------------
Batch: 0
-------------------------------------------


26/11/05 23:07:21 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 199208 milliseconds
[Stage 3:==>  (2 + 2) / 4][Stage 5:>    (0 + 2) / 4][Stage 6:>    (0 + 0) / 3]3]

+---------+----------+-----------+----------+------------+--------+------------+-----------------+--------+-----------+-------------------+
|visit_id |patient_id|facility_id|city      |status      |priority|wait_minutes|department       |desk_id |transfer_to|event_time         |
+---------+----------+-----------+----------+------------+--------+------------+-----------------+--------+-----------+-------------------+
|V-5698472|P-04339   |F-020      |Rabat     |DISCHARGED  |3       |NULL        |Pédiatrie        |NULL    |NULL       |2026-11-06 00:32:23|
|V-5698462|P-04804   |F-021      |Marrakech |CARE_STARTED|2       |44          |Médecine générale|NULL    |NULL       |2026-11-06 00:32:24|
|V-5699116|P-00474   |F-001      |Casablanca|ARRIVED     |NULL    |NULL        |Traumatologie    |NULL    |NULL       |2026-11-06 00:32:28|
|V-5699032|P-01251   |F-011      |Rabat     |TRIAGED     |3       |NULL        |Cardiologie      |T-RAB-02|NULL       |2026-11-06 00:32:32|
|V-5698946|P-03433  

26/11/05 23:07:52 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 256654 milliseconds
                                                                                

... (58 outputs left out)

+---------+----------+-----------+------+------------+--------+------------+-----------------+--------+-----------+-------------------+
|visit_id |patient_id|facility_id|city  |status      |priority|wait_minutes|department       |desk_id |transfer_to|event_time         |
+---------+----------+-----------+------+------------+--------+------------+-----------------+--------+-----------+-------------------+
|V-5701597|P-02409   |F-035      |Tanger|TRIAGED     |5       |NULL        |Médecine générale|T-TAN-10|NULL       |2026-11-06 03:15:36|
|V-5701606|P-02692   |F-040      |Tanger|TRIAGED     |5       |NULL        |Pédiatrie        |T-TAN-20|NULL       |2026-11-06 03:15:40|
|V-5700599|P-04221   |F-013      |Rabat |CARE_STARTED|5       |69          |Gynécologie      |NULL    |NULL       |2026-11-06 03:15:43|
|V-5700724|P-02035   |F-013      |Rabat |CARE_STARTED|3       |62          |Cardiologie      |NULL    |NULL       |2026-11-06 03:15:44|
|V-5701227|P-04683   |F-012      |Rabat |CARE_ST

26/11/05 23:12:34 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 15000 milliseconds, but spent 48019 milliseconds
26/11/05 23:12:36 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 18556 milliseconds
                                                                                

-------------------------------------------
Batch: 10
-------------------------------------------


[Stage 53:>                                                         (0 + 4) / 4]

In [8]:
for q in spark.streams.active:
    q.stop()

26/11/05 23:12:48 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: ConsoleWriter[numRows=25, truncate=false]] is aborting.
26/11/05 23:12:48 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 4, writer: ConsoleWriter[numRows=25, truncate=false]] aborted.


+---------+----------+-----------+----------+------------+--------+------------+-------------+--------+-----------+-------------------+
|visit_id |patient_id|facility_id|city      |status      |priority|wait_minutes|department   |desk_id |transfer_to|event_time         |
+---------+----------+-----------+----------+------------+--------+------------+-------------+--------+-----------+-------------------+
|V-5701711|P-02307   |F-003      |Casablanca|TRIAGED     |5       |NULL        |Chirurgie    |T-CAS-06|NULL       |2026-11-06 03:22:30|
|V-5701587|P-00579   |F-001      |Casablanca|CARE_STARTED|4       |14          |Pédiatrie    |NULL    |NULL       |2026-11-06 03:22:32|
|V-5701769|P-01213   |F-021      |Marrakech |TRIAGED     |3       |NULL        |Pneumologie  |T-MAR-01|NULL       |2026-11-06 03:22:33|
|V-5700760|P-02063   |F-012      |Rabat     |TRANSFERRED |4       |NULL        |Pédiatrie    |NULL    |F-011      |2026-11-06 03:22:38|
|V-5701790|P-01535   |F-021      |Marrakech |TRI

26/11/05 23:12:49 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 5, writer: ConsoleWriter[numRows=8, truncate=false]] is aborting.
26/11/05 23:12:49 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 5, writer: ConsoleWriter[numRows=8, truncate=false]] aborted.
26/11/05 23:12:49 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 12728 milliseconds


In [9]:
q_counts.stop()
q_wm.stop()
for q in spark.streams.active:
    print(q.name, "|", q.status["message"])

In [10]:
import json
p = q_console.lastProgress
print(json.dumps({k: p[k] for k in ("batchId", "numInputRows", "inputRowsPerSecond", "processedRowsPerSecond")}, indent=2) if p else "pas encore de batch")
p = q_bronze.lastProgress
print(p["batchId"], p["numInputRows"] if p else None)

{
  "batchId": 10,
  "numInputRows": 364,
  "inputRowsPerSecond": 19.532088430993777,
  "processedRowsPerSecond": 28.605108055009826
}
7 766


In [11]:
q_console.stop()

q_counts = (counts.writeStream
    .queryName("counts_city_priority")
    .outputMode("complete")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 25)
    .trigger(processingTime="15 seconds")
    .start())

26/11/05 23:13:58 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c9be1c63-9b22-4424-b075-8181d71c92ca. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/05 23:13:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [12]:
q_counts.stop()

q_wm = (wm.writeStream
    .queryName("watermark_demo")
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 8)
    .trigger(processingTime="15 seconds")
    .start())

26/11/05 23:14:00 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-5d268c88-19e2-4928-b7a1-25600029b6f6. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/05 23:14:00 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/05 23:14:01 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Casablanca|2       |1    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Rabat     |3       |1    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Marrakech |4       |1    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Rabat     |2       |1    |
+------------------------------------------+----------+--------+-----+



[Stage 60:>                                                         (0 + 4) / 4]

-------------------------------------------
Batch: 1
-------------------------------------------


+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Rabat     |4       |5    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Marrakech |3       |1    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Casablanca|5       |1    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Rabat     |3       |6    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Tanger    |1       |1    |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Tanger    |4       |1    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Marrakech |1       |1    |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Tanger    |5       |2    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Tanger    |5       |1    |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Tanger    |4       |8    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Casablanca|5       |2    |
|{2026-11-06 03:50:00, 2026-11-06 04:00:00}|Marrakech |3       |3    |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Rabat     |3       |11   |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Marrakech |4       |4    |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Tanger    |5       |7    |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Casablanca|2       |4    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



... (3 outputs left out)

26/11/05 23:15:02 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 15000 milliseconds, but spent 17012 milliseconds
[Stage 66:===========================================>              (3 + 1) / 4]

-------------------------------------------
Batch: 4
-------------------------------------------


+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Casablanca|5       |5    |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Rabat     |2       |4    |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Tanger    |5       |7    |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Casablanca|4       |9    |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Tanger    |3       |12   |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Marrakech |4       |10   |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Casablanca|3       |13   |
|{2026-11-06 04:00:00, 2026-11-06 04:10:00}|Tanger    |5       |9    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Rabat     |2       |5    |
|{2026-11-06 04:20:00, 2026-11-06 04:30:00}|Tanger    |3       |3    |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Casablanca|1       |1    |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Marrakech |5       |3    |
|{2026-11-06 04:20:00, 2026-11-06 04:30:00}|Marrakech |2       |1    |
|{2026-11-06 04:20:00, 2026-11-06 04:30:00}|Rabat     |4       |1    |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Casablanca|4       |11   |
|{2026-11-06 04:10:00, 2026-11-06 04:20:00}|Rabat     |3       |7    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



In [13]:
for q in spark.streams.active:
    q.stop()for q in spark.streams.active:
    q.stop()

In [14]:
import time
for q in spark.streams.active:
    if q.name == "watermark_demo":
        q.stop()

ck = f"s3a://lake/checkpoints/watermark_demo_{int(time.time())}"
q_wm = (wm.writeStream
    .queryName("watermark_demo")
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 8)
    .option("checkpointLocation", ck)
    .trigger(processingTime="15 seconds")
    .start())

26/11/05 23:15:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/05 23:15:53 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
[Stage 70:=============================>                            (2 + 2) / 4]

-------------------------------------------
Batch: 0
-------------------------------------------


+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Casablanca|4       |1    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Rabat     |4       |1    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Rabat     |3       |3    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Tanger    |3       |2    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Tanger    |1       |1    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Rabat     |5       |1    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Casablanca|2       |1    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Marrakech |3       |4    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



26/11/05 23:16:16 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 15000 milliseconds, but spent 24390 milliseconds
                                                                                

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+---------+--------+-----+
|window                                    |city     |priority|count|
+------------------------------------------+---------+--------+-----+
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Marrakech|2       |4    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech|4       |2    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Rabat    |3       |1    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Rabat    |4       |2    |
|{2026-11-06 04:20:00, 2026-11-06 04:30:00}|Rabat    |4       |1    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech|5       |2    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech|1       |1    |
|{2026-11-06 04:30:00, 2026-11-06 04:40:00}|Tanger   |4       |8    |
+------------------------------------------+---------+--------+-----+
only showing top 8 rows



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |4       |11   |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Rabat     |3       |3    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Casablanca|2       |1    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |5       |9    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |2       |5    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |1       |2    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Rabat     |4       |4    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |3       |9    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |4       |15   |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Casablanca|2       |3    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Rabat     |3       |4    |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Rabat     |5       |4    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |5       |11   |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |2       |7    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Rabat     |4       |7    |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Tanger    |5       |1    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



26/11/05 23:17:01 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 15000 milliseconds, but spent 16914 milliseconds
                                                                                

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------+--------+-----+
|window                                    |city      |priority|count|
+------------------------------------------+----------+--------+-----+
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Tanger    |5       |5    |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Rabat     |5       |6    |
|{2026-11-06 04:40:00, 2026-11-06 04:50:00}|Marrakech |2       |9    |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Marrakech |5       |4    |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Rabat     |2       |4    |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Rabat     |4       |6    |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Casablanca|4       |11   |
|{2026-11-06 04:50:00, 2026-11-06 05:00:00}|Casablanca|5       |1    |
+------------------------------------------+----------+--------+-----+
only showing top 8 rows



In [15]:
for q in spark.streams.active:
    q.stop()

In [16]:
import json
for q in spark.streams.active:
    print(q.name, "|", q.status["message"])
p = q_wm.lastProgress
if p:
    print(json.dumps(p["eventTime"], indent=2))
    print("Lignes ignorées :", [s.get("numRowsDroppedByWatermark") for s in p["stateOperators"]])

{
  "avg": "2026-11-06T04:53:32.223Z",
  "max": "2026-11-06T04:56:43.000Z",
  "min": "2026-11-06T04:36:35.000Z",
  "watermark": "2026-11-06T04:41:19.000Z"
}
Lignes ignorées : [0]
